# Лабораторная работа № 3.1
---
«Построение веб-приложения модели машинного обучения в Streamlit»

Рязанцев Георгий Борисович

## 0. Состав работы и запуск

Streamlit запускает приложения из файлов `.py` (`streamlit run файл.py`), запустить его из ноутбука `.ipynb` нельзя, поэтому код приложения вынесен в отдельные файлы. В ноутбуке находится всё остальное: проверка окружения, проверка модели вне Streamlit, автоматические тесты страниц, запуск и описание расхождений с методичкой.

Файлы лежат в папке `done\lab_3_1`:

| Файл | Что это | Раздел методички |
|---|---|---|
| `lab3_1.py` | главная страница: классификация ирисов | основная часть (рис. 3.1–3.5) |
| `pages\1_Регрессия_2.3.py` | сравнение моделей регрессии (`LazyRegressor`, `Rural_prices.csv`) | самостоятельная работа, модель из работы 2.3 |
| `pages\2_Тексты_2.5.py` | классификатор новостных текстов (TF-IDF + KNN) | самостоятельная работа, модель из работы 2.5 |
| `pages\3_Цифры_2.6.py` | распознавание цифр (`GaussianNB`) | самостоятельная работа, модель из работы 2.6 |
| `lab3_1.ipynb` | этот ноутбук | — |

Используются данные и файлы из соседних работ: `done\Rural_prices.csv` (2.3), `done\lab_2_5\knn_text_classifier.joblib`, `done\lab_2_5\DATA\stop-words.txt`, `done\lab_2_5\telegram_bot.py` (2.5); цифры (2.6) берутся из `sklearn.datasets.load_digits`, файлов не требуют. Пути в приложении заданы только для Windows (обратные слэши) и считаются от расположения самого файла (`__file__`), поэтому не зависят от того, из какой папки запущен `streamlit`.

**Запуск.** В терминале (Anaconda Prompt или терминал VS Code) из папки `done\lab_3_1`:

```
python -m streamlit run lab3_1.py
```

или ячейкой запуска в конце ноутбука.

## 1. Проверка окружения и файлов

In [ ]:
import os
import sys
import importlib.metadata as md

NB_DIR = os.getcwd()                      # ноутбук лежит в ...\done\lab_3_1
DONE_DIR = os.path.dirname(NB_DIR)        # ...\done

for pkg in ['streamlit', 'pandas', 'scikit-learn', 'matplotlib', 'seaborn', 'lazypredict', 'natasha', 'telethon', 'joblib']:
    try:
        print('%-13s %s' % (pkg, md.version(pkg)))
    except md.PackageNotFoundError:
        print('%-13s не установлена' % pkg)
print()

files = [
    NB_DIR + '\\lab3_1.py',
    NB_DIR + '\\pages\\1_Регрессия_2.3.py',
    NB_DIR + '\\pages\\2_Тексты_2.5.py',
    NB_DIR + '\\pages\\3_Цифры_2.6.py',
    DONE_DIR + '\\Rural_prices.csv',
    DONE_DIR + '\\lab_2_5\\knn_text_classifier.joblib',
    DONE_DIR + '\\lab_2_5\\telegram_bot.py',
    DONE_DIR + '\\lab_2_5\\DATA\\stop-words.txt',
    DONE_DIR + '\\lab_2_5\\SECRET\\BOT_TOKEN.txt',
    DONE_DIR + '\\lab_2_5\\SECRET\\MT_PROXY_SECRET.txt',
    DONE_DIR + '\\lab_2_5\\SECRET\\TELEGRAM_API.txt',
]
for f in files:
    print('есть ' if os.path.exists(f) else 'НЕТ  ', f)

**Что проверяется.** Установлены ли `streamlit` и библиотеки трёх моделей (`lazypredict` для страницы 2.3, `natasha` и `telethon` для страницы 2.5) и лежат ли все файлы на своих местах. Файлы из `SECRET` приложению нужны не для подключения к Telegram, а потому что `telegram_bot.py` читает их при импорте (см. раздел 5).

[УТОЧНИТЬ ПО РЕЗУЛЬТАТАМ ЗАПУСКА: версия Streamlit; все ли файлы найдены.]

## 2. Основная часть: классификация ирисов (рис. 3.1–3.5)

**Расхождения методички с кодом и их влияние.** Код `lab3_1.py` повторяет код методички (импорты, заголовок, боковая панель, функция `user_input_features`, модель `RandomForestClassifier`, вывод четырёх блоков). Отличия:

| № | В методичке | Что сделано | Почему и как влияет |
|---|---|---|---|
| 1 | Запуск командой «Streamlit run путь к файлу» в Anaconda Prompt | `python -m streamlit run lab3_1.py` из папки `done\lab_3_1` | Название команды пишется строчными (`streamlit`). Вариант `python -m` запускает Streamlit тем же интерпретатором, что и ноутбуки, и работает, даже если `streamlit` не добавлен в `PATH` |
| 2 | Функция `user_input_features()` вызывается дважды: сначала с заголовком «Выбранные параметры пользователем», затем во втором блоке снова `df = user_input_features()` с заголовком `User Input parameters` | Вызов оставлен один, заголовок один (русский, как на рис. 3.2–3.4) | Блоки на стр. 76–77 методичка показывает как два шага построения приложения, но если собрать их в один файл, четыре ползунка с одинаковыми параметрами создаются дважды, а Streamlit не допускает двух одинаковых виджетов без явного ключа (ошибка `DuplicateWidgetID`). На рисунках методички таблица параметров выводится один раз: рис. 3.2–3.4 получены без дубля |
| 3 | В коде подписи ползунков английские (`'Sepal length'` …), на рис. 3.2–3.4 — русские | Взяты русские подписи («Длина чашелистика», «Ширина чашелистика», «Длина лепестка», «Ширина лепестка») | Совпадение с рисунками; диапазоны и значения по умолчанию ползунков (4,3–7,9; 2,0–4,4; 1,0–6,9; 0,1–2,5; 5,4 / 3,4 / 1,3 / 0,2) взяты из кода методички без изменений. Названия столбцов таблицы `df` остаются английскими (`sepal_length` …), как на рис. 3.2 |
| 4 | `clf = RandomForestClassifier()`, `clf.fit(X, Y)` в основном теле скрипта | `RandomForestClassifier(random_state=0)`, обучение вынесено в функцию с `@st.cache_resource` | Streamlit выполняет весь скрипт заново при каждом изменении ползунка, поэтому лес обучался бы при каждом движении. Без `random_state` вероятности для пограничных объектов от обучения к обучению меняются (проверка в ячейке ниже). Теперь модель обучается один раз и результат воспроизводим. Для типичных точек (например, значений по умолчанию: setosa с вероятностью 1) результат тот же |
| 5 | `X = iris.data` (массив без названий), а `predict(df)` вызывается с таблицей | `X = pd.DataFrame(iris.data, columns=df.columns)` | Модель, обученная на массиве, при прогнозе по таблице выдаёт предупреждение sklearn («X has feature names, but … was fitted without feature names»). Порядок признаков в `iris.data` (длина и ширина чашелистика, длина и ширина лепестка) совпадает с порядком столбцов `df`, поэтому ответ не меняется |
| 6 | — | добавлены `st.set_page_config(page_title=...)` и подпись в боковой панели `st.sidebar.caption(...)` | Название вкладки браузера и подсказка, где находятся страницы самостоятельной работы; на расчёт не влияют |

Остальное (блоки «Возможная классификация и индекс вида», «Прогноз», «Вероятность прогноза», комментарий `#st.write(prediction)`) совпадает с методичкой. Рисунок 3.4 в методичке подписан «Рисунок 4».

In [ ]:
# Проверка модели из lab3_1.py вне Streamlit (те же данные и параметры)
import itertools
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.ensemble import RandomForestClassifier

iris = datasets.load_iris()
cols = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
X = pd.DataFrame(iris.data, columns=cols)
Y = iris.target

# 1) диапазоны ползунков методички и минимумы/максимумы данных
print('Диапазоны данных (ползунки в методичке: 4.3-7.9, 2.0-4.4, 1.0-6.9, 0.1-2.5):')
print(pd.DataFrame({'min': X.min(), 'max': X.max()}).to_string())
print()

# 2) значения по умолчанию (рис. 4: setosa с вероятностью 1)
clf = RandomForestClassifier(random_state=0).fit(X, Y)
default = pd.DataFrame([[5.4, 3.4, 1.3, 0.2]], columns=cols)
print('По умолчанию:', iris.target_names[clf.predict(default)][0], '| вероятности:', clf.predict_proba(default)[0].round(2))
print()

# 3) почему нужен random_state: 20 обучений без random_state для пограничной точки
border = pd.DataFrame([[6.0, 2.9, 4.8, 1.6]], columns=cols)
runs = np.array([RandomForestClassifier().fit(X, Y).predict_proba(border)[0] for _ in range(20)])
print('Пограничная точка', border.values[0].tolist(), '- вероятность versicolor в 20 обучениях без random_state: от %.2f до %.2f' % (runs[:, 1].min(), runs[:, 1].max()))
print()

# 4) точка, которую модель относит к versicolor с вероятностью не менее 0.99 (рис. 3.5 методички)
grid = pd.DataFrame(list(itertools.product(np.arange(4.3, 7.91, 0.5), np.arange(2.0, 4.41, 0.4),
                                           np.arange(1.0, 6.91, 0.5), np.arange(0.1, 2.51, 0.3))), columns=cols).round(1)
proba = clf.predict_proba(grid)
versicolor_points = grid[proba[:, 1] >= 0.99]
print('Точек сетки с вероятностью versicolor >= 0.99:', len(versicolor_points))
vers_point = [float(v) for v in versicolor_points.iloc[0]]
print('Пример (ползунки):', vers_point, '| вероятности:', clf.predict_proba(pd.DataFrame([vers_point], columns=cols))[0].round(2))

**Результат.** Диапазоны ползунков методички совпадают с минимумами и максимумами данных. При значениях по умолчанию модель относит цветок к `setosa` с вероятностью 1 (рис. 4 методички). Пример для рис. 3.5 методички (versicolor с вероятностью 0,99) воспроизводится на любой точке из списка выше.

[УТОЧНИТЬ ПО РЕЗУЛЬТАТАМ ЗАПУСКА: разброс вероятности versicolor для пограничной точки без random_state (в моём прогоне на другой версии sklearn: 0,77–0,91 за 20 обучений) и найденная точка с вероятностью ≥ 0,99.]

In [ ]:
# Автоматический тест главной страницы (Streamlit выполняет lab3_1.py без браузера)
from streamlit.testing.v1 import AppTest

at = AppTest.from_file(NB_DIR + '\\lab3_1.py', default_timeout=60).run()
print('Исключений:', len(at.exception), [e.value for e in at.exception])
print('Ползунки:', [(s.label, s.value) for s in at.sidebar.slider])
print('Заголовки:', [s.value for s in at.subheader])
tables = [d.value for d in at.dataframe]
print('Прогноз:', tables[2].iloc[0, 0], '| вероятности:', tables[3].iloc[0].round(2).tolist())

# те же ползунки, что в точке versicolor из предыдущей ячейки
for slider, value in zip(at.sidebar.slider, vers_point):
    slider.set_value(value)
at.run()
tables = [d.value for d in at.dataframe]
print('После изменения ползунков:', vers_point, '->', tables[2].iloc[0, 0], tables[3].iloc[0].round(2).tolist())

**Что проверяет ячейка.** `AppTest` выполняет скрипт приложения без браузера: проверяется, что нет исключений, ползунков четыре, выводятся четыре заголовка и таблицы и что изменение ползунков меняет прогноз. [УТОЧНИТЬ ПО РЕЗУЛЬТАТАМ ЗАПУСКА: прогноз и вероятности по умолчанию (ожидается setosa, [1, 0, 0]) и после изменения ползунков (ожидается versicolor с вероятностью ≥ 0,99).]

## 3. Самостоятельная работа: многостраничное приложение с тремя моделями

**Выбор моделей.** Выбраны модели из работ 2.3 (регрессия), 2.5 (классификация текстов) и 2.6 (распознавание цифр) — по одной на каждый основной тип данных (таблица, текст, изображение).

**Как построено многостраничное приложение.** Методичка отсылает к документации Streamlit «create-a-multipage-app». В ней описаны два способа: папка `pages` рядом с главным файлом (классический) и явная навигация `st.navigation` / `st.Page` в новых версиях. Выбран классический: он работает без дополнительного кода, не привязан к версии Streamlit, а порядок и название страниц в меню задаются именами файлов (`1_Регрессия_2.3.py` → пункт «Регрессия 2.3»). Главный файл `lab3_1.py` одновременно является страницей с ирисами (основная часть работы), страницы самостоятельной работы лежат в `pages`.

**Общее для страниц.** Каждая страница сама находит файлы по Windows-путям от своего расположения (`pages\<файл>.py` → `lab_3_1` → `done`), тяжёлые вычисления кэшируются (`st.cache_data`, `st.cache_resource`), чтобы они не повторялись при каждом действии пользователя, а при нехватке файла или библиотеки выводится понятное сообщение вместо трассировки.

### 3.1. Страница «Регрессия 2.3»

Показывает данные `Rural_prices.csv`, строит ~40 моделей `LazyRegressor`, сравнивает обучающую и тестовую выборки, рисует диаграмму выбранного показателя (R², RMSE, время). Расхождения с работой 2.3 в ноутбуке:

- **Кнопка запуска.** Построение моделей занимает от нескольких секунд до минуты, поэтому сначала выбираются параметры, затем нажимается «Построить модели». Результат кэшируется по сочетанию параметров.
- **`random_state` разбиения** по умолчанию 42 (в методичке 2.3 — `None`). Из-за кэширования и перерисовки страницы разбиение должно быть стабильным, иначе результаты менялись бы от случайности, а не от действий пользователя; значение можно изменить в боковой панели, как и долю тестовой выборки (по умолчанию 0,2, как в методичке).
- **Показан только основной результат работы 2.3:** таблица и диаграммы по обучающей и тестовой выборкам. Проверка на 10 разбиениях в приложение не переносилась (это отдельная долгая операция).
- **Шкала RMSE** ограничена пределом, который можно менять (по умолчанию 75 000, как в методичке); модели с ошибкой выше предела перечисляются под диаграммой. Для R² ось от 0 до 1, модели с отрицательным R² перечисляются под диаграммой (как в ноутбуке 2.3: на графике они не видны).
- Состав моделей зависит от установленных пакетов (в окружении, где строилась работа 2.3, нет `xgboost` и `lightgbm`).

### 3.2. Страница «Тексты 2.5»

Принимает текст новости (или один из трёх готовых примеров) и показывает рубрику и доли голосов ближайших соседей. Использует функции из `telegram_bot.py` без переписывания, как в ноутбуке 2.5 (`telegram_bot.predict_rubric`, `telegram_bot.rubrics`, `telegram_bot.model`, `telegram_bot.vectorizer`). Особенности:

- **Импорт `telegram_bot`.** При импорте файл читает три файла из `lab_2_5\SECRET`, создаёт объект `TelegramClient` (подключения к Telegram при этом нет, оно выполняется только в `main()`, которая запускается лишь при запуске файла напрямую) и загружает модели `natasha`, `joblib` и список стоп-слов. Все пути в `telegram_bot.py` считаются от его расположения (`BASE_DIR`), поэтому импорт из другой папки работает. Страница импортирует его один раз (`@st.cache_resource`), поэтому загрузка при первом открытии занимает заметное время.
- **`os.chdir` на время импорта.** Telethon создаёт файл сессии `test_bot.session` в текущей папке; чтобы он не появлялся в `lab_3_1`, текущая папка на время импорта заменяется на `lab_2_5` (там такой файл уже есть) и затем возвращается.
- **Цикл событий.** Страница перед импортом создаёт цикл событий asyncio, если в потоке Streamlit его нет: Telethon при создании клиента его запрашивает. Это страховка, не проверенная на вашей версии Streamlit, поэтому ниже есть автоматический тест страницы.
- **Те же ограничения, что у бота.** Текст короче 5 слов отклоняется, если в тексте нет слов из словаря модели, выводится сообщение (как в боте).
- **Не нужно подключение к Telegram и прокси.** Если на странице появляется ошибка `database is locked`, закройте работающий бот, использующий тот же файл сессии.

### 3.3. Страница «Цифры 2.6»

Обучает `GaussianNB` на `load_digits` так же, как в ноутбуке 2.6 (`train_test_split(..., random_state=0)`, обучение на 1347 изображениях), и показывает: accuracy на тестовой выборке и матрицу несоответствий; одно изображение с ответом модели и вероятностями цифр (ползунок по номеру); ряд цифр от 0 до 9 (самостоятельная работа 2.6). Результаты должны совпадать с ноутбуком 2.6: accuracy на тесте 0,8333; на ряде цифр 0–9 верно 8 из 10 (ошибки на цифрах 2 → 8 и 5 → 3). Модель обучается один раз (`@st.cache_resource`).

In [ ]:
# Автоматические тесты страниц (выполнение без браузера)
import warnings
warnings.filterwarnings('ignore')

# --- страница 2.3: нажатие кнопки «Построить модели»
at = AppTest.from_file(NB_DIR + '\\pages\\1_Регрессия_2.3.py', default_timeout=600).run()
print('Регрессия 2.3: исключений до нажатия:', len(at.exception), '| сообщение:', [i.value for i in at.info])
at.button(key='reg_build').click().run()
print('Регрессия 2.3: исключений после нажатия:', len(at.exception), [e.value for e in at.exception])
print('  ', [s.value for s in at.success])
print('  таблиц:', len(at.dataframe), '| ошибок:', [e.value for e in at.error])

In [ ]:
# --- страница 2.5: текст из готового примера
sport = 'Футбольная команда выиграла матч у соперника со счётом два один и вышла в финал чемпионата. Тренер похвалил игроков, а нападающий забил победный гол на последней минуте.'
at = AppTest.from_file(NB_DIR + '\\pages\\2_Тексты_2.5.py', default_timeout=600).run()
print('Тексты 2.5: исключений при загрузке:', len(at.exception), [e.value for e in at.exception], '| ошибок:', [e.value for e in at.error])
at.text_area(key='news_text').input(sport).run()
at.button(key='classify').click().run()
print('  ', [s.value for s in at.success], '| предупреждений:', [w.value for w in at.warning])
if len(at.dataframe):
    print(at.dataframe[0].value.to_string())

In [ ]:
# --- страница 2.6: одно изображение и ряд цифр 0-9
at = AppTest.from_file(NB_DIR + '\\pages\\3_Цифры_2.6.py', default_timeout=120).run()
print('Цифры 2.6: исключений:', len(at.exception), [e.value for e in at.exception])
print('  метрики:', [(m.label, m.value) for m in at.metric])
at.radio(key='digits_mode').set_value(at.radio(key='digits_mode').options[1]).run()
print('  режим «ряд 0-9»: исключений:', len(at.exception))
print(at.dataframe[0].value.to_string())
print('  ', [w.value for w in at.markdown][-1])

**Результат тестов.** [УТОЧНИТЬ ПО РЕЗУЛЬТАТАМ ЗАПУСКА: для каждой страницы — число исключений (ожидается 0); страница 2.3 — лучшая модель на тесте и число таблиц; страница 2.5 — рубрика для спортивного примера (ожидается «спорт») и доли голосов; страница 2.6 — accuracy (ожидается 0.833) и «верно распознано: 8 из 10».]

Тесты `AppTest` выполняют только код страницы, но не отображение в браузере, поэтому после них приложение нужно один раз запустить и просмотреть вручную (следующая ячейка).

## 4. Запуск приложения

In [ ]:
import subprocess

# Запуск Streamlit в отдельном процессе; в браузере откроется http://localhost:8501
proc = subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', 'lab3_1.py'], cwd=NB_DIR)
print('Приложение запущено (процесс %d). Адрес: http://localhost:8501' % proc.pid)

In [ ]:
# Остановка приложения
proc.terminate()
print('Приложение остановлено')

## 5. Ответы на вопросы для самоконтроля

1. **Задача автоматизации и отображения результатов построения моделей.** Результаты анализа и модели оформляются в виде веб-приложения, которым можно пользоваться без запуска кода: пользователь меняет параметры или загружает данные, а приложение пересчитывает прогноз и показывает таблицы и графики. При обновлении данных достаточно перезапустить приложение.
2. **Назначение Streamlit.** Быстро создавать веб-приложения для анализа данных и машинного обучения на чистом Python, без HTML и JavaScript: виджеты, таблицы, графики, многостраничность, публикация через Streamlit Cloud.
3. **Запуск приложения.** `streamlit run путь_к_файлу.py` (в Windows надёжнее `python -m streamlit run файл.py`); приложение открывается в браузере по адресу `http://localhost:8501`.
4. **Заголовки, боковые панели, таблицы, рисунки.** `st.write` / `st.title` / `st.header` / `st.subheader` — текст и заголовки; `st.sidebar.<виджет>` — элементы на боковой панели (ползунки `st.sidebar.slider`, заголовок `st.sidebar.header`); `st.write(df)` и `st.dataframe(df)` — таблицы; `st.pyplot(fig)`, `st.bar_chart`, `st.image` — рисунки и графики.
5. **Особенности многостраничного приложения.** Страницы — отдельные файлы в папке `pages` рядом с главным файлом (или `st.Page` и `st.navigation` в новых версиях); порядок и названия берутся из имён файлов; состояние между страницами передаётся через `st.session_state`; тяжёлые операции кэшируются, потому что скрипт страницы выполняется заново при каждом действии пользователя.

## Итоги сверки с методичкой

| Что | В методичке | Реальные значения / комментарий |
|---|---|---|
| Установка и запуск | `pip install Streamlit`, команда «Streamlit run путь» | `python -m streamlit run lab3_1.py` из `done\lab_3_1` |
| Блок `user_input_features()` | вызывается дважды | вызов один: два одинаковых виджета в одном скрипте приводят к ошибке |
| Подписи ползунков | в коде английские, на рисунках русские | русские, как на рисунках |
| Модель | `RandomForestClassifier()` обучается при каждом выполнении скрипта | `random_state=0` и кэширование (`st.cache_resource`) |
| Признаки модели | массив `iris.data` | таблица с теми же названиями столбцов, что у `df` (без предупреждения sklearn) |
| Результат при параметрах по умолчанию (рис. 4) | setosa, вероятность 1 | совпадает: [УТОЧНИТЬ] |
| Versicolor с вероятностью 0,99 (рис. 3.5) | указано без значений ползунков | воспроизведено на точке: [УТОЧНИТЬ] |
| Самостоятельная работа | многостраничное приложение с тремя моделями, ссылка на документацию | классический механизм `pages`; страницы: регрессия 2.3, тексты 2.5, цифры 2.6; тесты страниц через `AppTest` |
| Пути к файлам | не оговариваются | только Windows, от расположения файла (`__file__`) |
| Импорт `telegram_bot` | не рассматривается | импорт без запуска бота, с заменой текущей папки на время импорта |